## Status Check

In [ ]:
!nvidia-smi

## SYSTEM VAR

In [ ]:
DEFAULT_MAX_TOKENS = 512
QWEN3=True
SEED=42

## Library Import

In [ ]:
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration, Qwen2_5_VLForConditionalGeneration
import torch

from sam2.sam2_image_predictor import SAM2ImagePredictor
from sam2.build_sam import build_sam2

# from sam3.model_builder import build_sam3_image_model
# from sam3.model.sam3_image_processor import Sam3Processor
# from sam3.model.box_ops import box_xywh_to_cxcywh
# from sam3.visualization_utils import draw_box_on_image, normalize_bbox, plot_results

import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import json
import matplotlib.patches as patches
from dataclasses import dataclass, field
from collections import Counter

torch.cuda.is_available()

## Default Function Init

In [ ]:
def set_seed(seed):
    os.environ['PYTHONHASHSEED'] = str(seed) 
    random.seed(seed)  
    np.random.seed(seed)  
    torch.manual_seed(seed)  
    torch.cuda.manual_seed(seed)  
    torch.cuda.manual_seed_all(seed)  
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed(SEED)

In [ ]:
def encode_single_mask(mask):
    """
    Input:
        mask: [1000, 1000] 또는 [B, 1000, 1000] 형태의 bool 타입의 np.ndarray 혹은 torch.Tensor
    Returns:
        RLE 문자열
    """
    if torch.is_tensor(mask):
        mask = mask.detach().cpu().numpy()
    # 픽셀을 일렬로 세움 (Flatten ; order='F')
    pixels = mask.T.flatten()

    pixels = np.concatenate([[False], pixels, [False]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]

    # 결과 문자열 생성 (예: "1 3 10 5")
    return ' '.join(str(x) for x in runs)


## Model Load

In [ ]:
MODEL_PATH = "/home/jjs2403/2026_bootcamp_02/models/Qwen3-VL-8B-Instruct"
qwen3_model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_PATH, 
    dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto", 
    attn_implementation="flash_attention_2"
)
qwen3_processor = AutoProcessor.from_pretrained(MODEL_PATH)


In [ ]:
# MODEL_PATH = "/home/jjs2403/2026_bootcamp_02/models/Qwen2.5-VL-7B-Instruct"

# qwen25_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
#     MODEL_PATH,
#     dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
#     device_map="auto",
#     attn_implementation="flash_attention_2",
# )

# qwen25_processor = AutoProcessor.from_pretrained(MODEL_PATH)


In [ ]:
SAM2_1_MODEL_PATH = "//home//jjs2403//2026_bootcamp_02//models//SAM2.1//weights//sam2.1_hiera_large.pt"
SAM2_1_MODEL_CFG_PATH = "//home//jjs2403//2026_bootcamp_02//models//SAM2.1//weights//sam2.1_hiera_l.yaml"

sam21_mask_predictor = SAM2ImagePredictor(build_sam2(SAM2_1_MODEL_CFG_PATH, SAM2_1_MODEL_PATH))

In [ ]:
# SAM3_MODEL_PATH = "/home/jjs2403/2026_bootcamp_summer/models/sam3/sam3.pt"
# device = 'cuda:0'
# sam3_model = build_sam3_image_model(
#     eval_mode=True,
#     checkpoint_path="/home/jjs2403/2026_bootcamp_summer/models/sam3/sam3.pt",
#     load_from_HF=False,
#     enable_segmentation=True,
#     enable_inst_interactivity=False,
#     compile=False,
# ).to(device)
# sam3_processor = Sam3Processor(sam3_model)


## Dataset Load

In [ ]:
class Task2Dataset(Dataset):
    """
    Dataset 클래스.

    Return 값은 다음과 같다.
    image_id: image번호(000__.png)
    image: PIL 원본이미지
    query: 해당 이미지의 쿼리
    """
    def __init__(self, data_path, query_path):
        self.data_path = data_path
        self.query_df =pd.read_csv(query_path)

    def __len__(self):
        return len(self.query_df)

    def __getitem__(self, idx):
        row =self.query_df.iloc[idx]
        image_id =int(row["ID"])
        query =row["Query"]
        image = Image.open(os.path.join(self.data_path, f"{image_id:05d}.png")).convert("RGB")

        return image_id, image, query # image는 그냥 PIL 이미지가 반환되니까 주의!!!

In [ ]:
BASE_PATH = "/home/jjs2403/2026_bootcamp_summer/_ASSIGNMENTS/competitions/competition2"

test_dataset = Task2Dataset(
    data_path="/home/jjs2403/2026_bootcamp_summer/_ASSIGNMENTS/competitions/competition2/imgs",
    query_path="/home/jjs2403/2026_bootcamp_summer/_ASSIGNMENTS/competitions/competition2/query.csv"
)

In [ ]:
def collate_fn(batch):
    return batch[0]

test_loader = DataLoader(test_dataset,batch_size=1,shuffle=False,collate_fn=collate_fn)

## Template&Parsing Function

In [ ]:
### @deprecated @fallback
### sub5 version(OLD)
def template_old(query): ##### DEFAULT TEMPLATE ####
    return f"""
Identify the target that answers the query or is referred to by the query in the image.

If the target can be identified directly, locate it directly.
If the query requires reasoning, counting, comparison, or solving, use the image information to determine the answer first and then locate the target.
Use only the reasoning needed for the query.

After determining the answer, locate that exact visual target in the image.
Before returning the result, verify that each bbox and point are on the object described by its concept, not on a nearby or related object.

Return only valid JSON in this format:
{{
    "description": "short description of the target",
    "targets": [
        {{
            "concept": "short visual noun phrase",
            "bbox_2d": [x1, y1, x2, y2],
            "point_2d": [x, y]
        }}
    ]
}}

The point must be inside the target.
If there are multiple targets, return all of them in the targets list.

Query: {query}
"""

In [ ]:
def template_classify(query): ## router용
    return f"""
Classify the following query into exactly one reasoning type below, based on what kind of reasoning process is needed to find the target it refers to.

A. Direct Grounding - the query names or describes the target explicitly, with no reasoning needed.
B. Indirect Clue / Figurative - the target is described indirectly, through metaphor, behavior, or implied visual clues rather than named directly.
C. Functional & Situational Commonsense - the query describes a situation, need, or condition, and the target must be found using common-sense knowledge of what object serves that function.
D. Comparative / Ranking - the query requires comparing multiple candidate objects on some attribute and selecting one based on an extreme value or rank (e.g., most, least, second-highest).
E. Quantitative / Logical Reasoning - the query requires an explicit calculation or step-by-step logical deduction, such as arithmetic, date/calendar computation, solving a logic puzzle (e.g., Sudoku), or reading a value from a chart or graph.
F. Factual / Domain Knowledge - the query requires recalling a specific fact (e.g., game rules, regulations, categorical facts) to determine the correct target, without needing multi-step calculation or deduction.

Return only valid JSON in this format:
{{
    "reason": "brief explanation of why this type was chosen",
    "type": "A"
}}

Query: {query}
"""

In [ ]:
def parse_query_type(output):
    """
    output 출력결과를 넣으면 qtype(A~F)과 reason을 반환하는 함수.
    단. Fallback시 None과 빈 String을 반환함.
    """
    output = output.replace("```json", "").replace("```", "").strip()

    try:
        start =output.find("{")
        end =output.rfind("}") + 1

        if start==-1 or end<=start:
            return None, ""

        result =json.loads(output[start:end])

        qtype =str(result.get("type", "")).strip().upper()
        reason =str(result.get("reason", "")).strip()

        if qtype not in ["A", "B", "C", "D", "E", "F"]:
            return None, reason

        return qtype, reason

    except:
        print("[ERROR] parse_query_type()")
        return None, ""

In [ ]:
TYPE_INSTRUCTIONS = {"A": (
        "Identify the exact target described by the query in the image and locate it directly. "
        "Do not add unnecessary reasoning steps because the query already names or describes the target explicitly."
    ),"B": (
        "The query refers to the target indirectly, through metaphor, behavior, or an implied clue rather than naming it directly. "
        "First interpret what real-world object, person, or state the description is actually pointing to, "
        "then locate that exact target in the image."
    ),"C": (
        "The query describes a situation, need, or condition. "
        "First identify exactly what condition is being described, then find the object in the image that functionally satisfies it, "
        "based on common-sense knowledge of how such objects are normally used in that situation. "
        "Prefer the ordinary and commonly expected use or behavior. "
        "Do not rely on rare, unusual, or merely technically possible edge cases."
    ),"D": (
        "The query requires comparing multiple candidate objects in the image along some attribute. "
        "First identify all relevant candidates, evaluate the attribute for each one, "
        "then select the one matching the specified rank or extreme (e.g., most, least, second-highest). "
        "Double-check the rank before finalizing and do not confuse an extreme value with a specific rank."
    ),"E": (
        "The query requires a calculation, counting, numerical or time-based filtering, or logical deduction. "
        "Evaluate the entire condition rather than only one local step. "
        "Perform the required calculation or deduction first, then independently check the result once. "
        "If the verification contradicts the first result, recalculate before locating the corresponding target in the image."
    ),"F": (
        "The query requires specific factual or domain knowledge (e.g., game rules, official regulations, categorical facts) "
        "to determine the correct target. "
        "Recall the relevant fact first, then apply that fact consistently to identify and locate the correct target in the image."
    ),}


def template_v2(query, qtype):
    if qtype is None:
        return template_old(query)
    return f"""
{TYPE_INSTRUCTIONS[qtype]}

Before solving the query, identify its exact segmentation requirements:
- the requested number or quantifier,
- every explicitly requested visual component,
- and every exclusion or negative condition.

Preserve all of these requirements exactly in the final targets list.
Do not omit one component when the query requests multiple components.
Do not include related or contextual objects that are not requested.

Before finalizing the JSON, check the targets list against those requirements.

Every explicitly requested visual component must be represented in the targets list.
If multiple spatially separate visible targets are requested, return a separate target entry for each one.
Do not merge separate requested targets into one bounding box.

Verify once that:
- no required target is missing,
- no excluded or unrequested target is included,
- and the number of returned targets matches the query whenever the query specifies a number.

If the query specifies an exclusion (e.g., "excluding X", "not Y", "other than Z"), make sure the final target does not fall under the excluded case.

Before returning the result, verify that each bbox and point are on the object described by its concept, not on a nearby or related object.

The semantic type of each target must exactly match what the query asks to segment.

If multiple spatially separate targets satisfy the query, return a separate target entry for each one.
Do not use one large bounding box to cover multiple separate targets.

Assume that at least one valid target exists in the image.
Do not return an empty "targets" list.
If your first interpretation finds no valid target, reconsider the query and image and select the best matching visible target.

Keep the reasoning concise and non-repetitive.
Use at most 4 short sentences.
Do not enumerate every candidate, cell, or item individually when a shorter summary is sufficient.
Always leave enough room to complete the "description" and "targets" fields.

Return only valid JSON in this format:
{{
    "reasoning": "concise reasoning process used to determine the target",
    "description": "short description of the target",
    "targets": [
        {{
            "concept": "short visual noun phrase",
            "bbox_2d": [x1, y1, x2, y2],
            "point_2d": [x, y]
        }}
    ]
}}

The point must be inside the target.

Query: {query}
"""

In [ ]:
def parse_output(output):
    """
    Qwen Output을 Json으로 Parsing하는 Funcion
    targets 리스트에 모든 BOX 저장
    """
    output=output.replace("```json", "").replace("```", "").strip()
    start=output.find("{")
    end=output.rfind("}")+1
    result = json.loads(output[start:end])
    description = result.get("description", "")
    targets = []

    for item in result["targets"]: ## ALL TARGET SAVED
        targets.append({
            "concept": item["concept"].strip(),
            "bbox": item["bbox_2d"],
            "point": item["point_2d"]
        })

    return description, targets

def parse_output_v2(output):
    """
    reasoning 필드가 포함된 2단계 출력을 파싱하는 함수.
    """
    output = output.replace("```json", "").replace("```", "").strip()
    start = output.find("{")
    end = output.rfind("}") + 1
    result = json.loads(output[start:end])
    reasoning = result.get("reasoning", "") # REASONING
    description = result.get("description", "") # DESCRIPTION
    targets = []

    for item in result["targets"]:
        targets.append({
            "concept": item["concept"].strip(),
            "bbox": item["bbox_2d"],
            "point": item["point_2d"]
        })

    return description, targets, reasoning

## Inference Function

In [ ]:
def qwen_get_response(model, processor, image, query, max_new_tokens=DEFAULT_MAX_TOKENS):
    model.eval()
    messages = [{
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": query},
            ],
        }]

    input_text = processor.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
    inputs = processor(text=[input_text],images=[image],return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(**inputs,max_new_tokens=max_new_tokens,do_sample=False)

    return processor.batch_decode(output_ids[:, inputs.input_ids.shape[1]:],skip_special_tokens=True)[0]

In [ ]:
def qwen_inference_v2(model, processor, image, query, max_retry=1): ## MAX RETRY는 웬만해선 1로 고정할것. 2해봤자 sampling 없어서 똑같음 ㅇㅇ
    """
    RETURN값
    output, description, targets, qtype, cls_reason, reasoning, status(FALLBACK or not)
    """
    ############# STAGE 1 ##################
    qtype = None # Question Type
    cls_reason = "" # Classfication Reason.

    for retry in range(max_retry):
        cls_output = qwen_get_response(model, processor,image,template_classify(query))
        qtype, cls_reason = parse_query_type(cls_output)
        if qtype is not None:
            break
        print(f"[WARN] Query Type Parsing Error - Retry {retry + 1}/{max_retry}")

    # Classfication fail
    if qtype is None:
        print("Query Type Classification Failed - Use Default Prompt")
        output = qwen_get_response(model, processor,image,template_old(query)) ## FALLBACK ##
        try:
            description, targets = parse_output(output)
            return output, description, targets, None, cls_reason, "", "classifier_fallback_success"
        except:
            print("[ERROR] A: Default Prompt JSON Parsing Failed")
            print("Raw Output:")
            print(output)
            return output, "", [], None, cls_reason, "", "classifier_fallback_failure" ### COMPELETE FAILURE ###

    ########## Stage 2 ###########
    output = "ERROR"
    for retry in range(max_retry):
        output = qwen_get_response(model, processor,image,template_v2(query, qtype),max_new_tokens=1024)
        try:
            description, targets, reasoning = parse_output_v2(output)
            return output, description, targets, qtype, cls_reason, reasoning, "complete_success"
        except:
            print(f"[ERROR] B: JSON Parsing Error - Retry {retry + 1}/{max_retry}")
            print("Raw Output:")
            print(output)

    #### FALLBACK 2 ###
    print("Specialized Prompt Failed - Use Default Prompt")
    output = qwen_get_response(model, processor,image,template_old(query)) ## FALLBACK ##
    try:
        description, targets = parse_output(output)
        return output, description, targets, qtype, cls_reason, "", "specialized_fallback_success"
    except:
        print("[ERROR] C: Default Prompt JSON Parsing Failed")
        print("Raw Output:")
        print(output)
        return output, "", [], qtype, cls_reason, "", "specialized_fallback_failure" ### COMPELETE FAILURE ###

## Functions for SAM

In [ ]:
def expand_bbox(bbox, width, height, scale=1.05):
    """
    BBOX를 확대
    """
    x1,y1,x2,y2=map(float,bbox)
    cx=(x1+x2)/2 # x - center of bbox
    cy=(y1+y2)/2 # y - center of bbox
    w=(x2-x1)*scale
    h=(y2-y1)*scale

    ## out of range 방지
    x1=max(0, cx-w/2)
    y1=max(0, cy-h/2)
    x2=min(width,cx+w/2)
    y2=min(height,cy+h/2)

    return [x1,y1,x2,y2]

def segment_targets_with_sam21(image, targets, bbox_scale=1.05):
    sam21_mask_predictor.set_image(np.array(image))
    final_mask=np.zeros((1000, 1000), dtype=bool)

    for target in targets:
        bbox = target["bbox"]
        point = target["point"]
        bbox = expand_bbox(bbox, width=image.width, height=image.height, scale=bbox_scale)

        input_box = np.array(bbox, dtype=np.float32)
        input_point = np.array([point])
        input_label = np.array([1])

        with torch.inference_mode():
            masks, scores, logits = sam21_mask_predictor.predict(
                point_coords=input_point,
                point_labels=input_label,
                box=input_box,
                multimask_output=True,
            )

        sorted_ind = np.argsort(scores)[::-1]
        masks = masks[sorted_ind]
        final_mask |= masks[0].astype(bool) # final mask 합치기

    return final_mask.astype(bool)

## Visualize Function

In [ ]:
def show_bbox_point(image, targets):
    fig, ax = plt.subplots()
    ax.imshow(image)

    for target in targets:
        bbox = target["bbox"]
        point = target["point"]

        x1, y1, x2, y2 = bbox
        x, y = point

        rect = patches.Rectangle(
            (x1, y1),
            x2 - x1,
            y2 - y1,
            linewidth=2,
            fill=False,
            edgecolor="red"
            # edgecolor="cyan"
        )

        ax.add_patch(rect)
        ax.scatter(x, y, s=30, color="cyan")

    plt.show()

In [ ]:
def show_mask(mask):
    plt.imshow(mask)
    plt.show()

## <------- 공용 부분 종료 EOSS ------->
** 공용부분 필요시 확장. 수정시 전체기록

## Pipeline

In [ ]:
masks=[] # Mask 결과 최종저장

# router 성능 측정변수 INIT
route_counts = Counter()
status_counts = Counter()
route_logs = []
masks = []

for i, (image_id, image, query) in enumerate(test_loader):
    # if i <10:
    #     continue
    # if i >= 20 :
    #     break
    # if not i in [4, 6, 11, 26, 35, 36, 59]:
    #     continue

    if QWEN3: ## ENV VARIABLE ##
        model=qwen3_model
        processor=qwen3_processor
    else:
        # model=qwen25_model
        # processor=qwen25_processor
        break 

    # 출력 / 설명문 / bbox and point 리스트 / Query Type(A~F) / 추론 / 상태(기록용)
    output, description, targets, qtype, cls_reason, reasoning, status = qwen_inference_v2(model,processor,image,query)
    route = qtype if qtype is not None else "DEFAULT"

    route_counts[route]+=1
    status_counts[status]+=1
    route_logs.append({
        "image_id": int(image_id),
        "query": query,
        "qtype": route,
        "status": status,
        "cls_reason": cls_reason,
        "num_targets": len(targets),
    })

    print(f"ID{image_id}: {query}")
    print(f"Query Type: {qtype} ({cls_reason})")
    print(f"Route Status: {status}")
    print(f"Reasoning: {reasoning}")
    print(f"Qwen Output: {output}")
    print("Description:", description)
    print("Targets:", targets)
    
    mask_sam21 = segment_targets_with_sam21(image,targets,bbox_scale=1.03)
    
    # mask_sam3 = segment_targets_with_sam3(
    #     image,
    #     targets,
    #     use_bbox=False
    # )

    # mask_sam3_bbox = segment_targets_with_sam3(
    #     image,
    #     targets,
    #     use_bbox=True
    # )

    show_bbox_point(image, targets)

    # print("SAM2.1")
    show_mask(mask_sam21)
    masks.append(mask_sam21)

    # print("SAM3 Concept Only")
    # show_mask(mask_sam3)

    # print("SAM3 Concept + BBox")
    # show_mask(mask_sam3_bbox)

## Save Submission

In [ ]:
!pwd

In [ ]:
submission = pd.read_csv("sample_submission2.csv")
# submission = pd.read_csv("/home/bootcamp2026_30/sample_submission2.csv") ### 출력이 안될시 절대경로로 수정(주피터에 샘플 업로드)
lst = []

for mask in masks:
    mask = np.asarray(mask).squeeze()
    mask = mask.astype(bool)
    encoded_mask = encode_single_mask(mask)
    lst.append(encoded_mask)

submission["label"] = lst
submission.to_csv("submission.csv", index=False)

In [ ]:
for key in ["A", "B", "C", "D", "E", "F", "DEFAULT"]:
    print(f"{key}: {route_counts[key]}")

In [ ]:
route_df = pd.DataFrame(route_logs)
route_df.to_csv("route_logs.csv", index=False)